# Machine Learning Project 1 — Chained Bike-Demand Pipeline

This notebook is the final presentation layer. Core implementation lives in `src/`.

## Phase 1 — Gradient Descent

### Expectation

_Write this **before** running the phase._

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data import compute_team_seed, load_data, seeded_split
from src.features import Phase1Features
from src.metrics import regression_metrics
from src.phase1_gradient_descent import (
    LinearRegressionGD,
    largest_curvature,
    load_phase1_artifact,
    save_phase1_artifact,
)
from src.visualization import plot_gd_runs, plot_shift_by_hour

# Team seed: first reproduce the brief's example, then compute ours.
assert compute_team_seed(["34521", "40218", "41190"]) == 41698
seed = compute_team_seed(config.TEAM_IDS)
assert seed == config.EXPECTED_SEED
print("Team seed:", seed)

# The only seeded split: 80% train / 20% validation.
train_all, test_X = load_data(config.TRAIN_PATH, config.TEST_PATH)
train_df, val_df = seeded_split(train_all, seed, config.VALIDATION_SIZE)
y_train = train_df[config.TARGET].to_numpy(dtype=float)
y_val = val_df[config.TARGET].to_numpy(dtype=float)
print(f"train: {len(train_df)} rows, validation: {len(val_df)} rows")

### Phase 1 feature choices and justification

`Phase1Features` (in `src/features.py`) builds every column below. Its one-hot levels and its scaler are fitted on the 8,708 training rows only, then applied unchanged to validation (and, at the end, to `test.csv`). The evidence is in the two cells that follow.

| Column | Phase 1 representation | Why |
|---|---|---|
| `hr` | one-hot, 23 dummies (00:00 is the reference) | Demand has two weekday commute peaks and a weekend midday hump. A numeric `hr` would force one straight line across the day; one-hot gives every hour its own level. |
| `weekday` | one-hot, 6 dummies (Sunday is the reference) | Days of the week differ in level; the numbers 0–6 are labels, not amounts. |
| `holiday` | 0/1 | Holidays are rare but can change the daily pattern. |
| `workingday` | **not used** | In our training rows `workingday` = (Mon–Fri) − `holiday` with 0 exceptions (holidays only fall on Mon–Fri). With the weekday dummies, `holiday` and the intercept it is an exact linear combination, so the weights would not be unique. We keep `weekday` + `holiday` because they carry strictly more (Saturday vs Sunday, Monday vs Friday); `workingday` can be rebuilt from them for Phase 2. |
| `weathersit` | one-hot, levels 2 and 3 (1 = clear is the reference); level 4 is merged into 3 | Our training portion has 0 rows with level 4 (the only labeled one fell into validation), so no coefficient can be learned for it. Level 4 is the most severe level, closest to 3; left unmapped, it would silently be treated as clear weather. |
| `mnth` | one-hot, 11 dummies (January is the reference) | Seasonal representation. Every month in our training rows has exactly one `season`, so `season` is a coarser copy of `mnth` and the two cannot be used together. Month did better on validation (R² 0.6823 vs 0.6805) and separates months that season groups together, e.g. January from March. |
| `season` | **not used** | Exact function of `mnth` here (see above). |
| `yr` | 0/1 | Demand grows from 2011 to 2012. |
| `temp`, `hum`, `windspeed` | numeric | Weather measurements. Phase 1 is linear in them; non-linear effects are Phase 2's job. |
| `atemp` | **not used** | Correlation 0.984 with `temp` in our training rows, so it is a near-copy; its verdict is tested properly in Phase 4. |
| `dteday` | used through `yr`, `mnth`, `weekday`; no extra column | Its year, month and weekday agree exactly with `yr`, `mnth` and `weekday` (0 mismatches each), so those columns already carry them. Day-of-month is unusable: training has only days 1–19 and `test.csv` only day 20. What remains is elapsed time (`days_since_start`). It adds only 0.0005 R² (0.6829 vs 0.6823), but it is almost a combination of `yr` and month: the condition number jumps from 56 to 8,084 and GD needs 36,407 updates instead of 512. Not worth it in Phase 1. |
| `instant` | **not used** | Row identifier; kept only to align the submission file. |
| `cnt` | target | Never part of X. |

**Scaling.** Every feature column (dummies included) is standardized with its training mean and standard deviation; the intercept column stays at 1. All gradient components are then on comparable scales, so one learning rate suits every weight, and the learning rate can be stated relative to the curvature of the loss.

In [ ]:
# Facts behind the encoding choices, checked on the training portion.
dates = pd.to_datetime(train_df["dteday"])
mon_to_fri = train_df["weekday"].between(1, 5).astype(int)
encoding_facts = pd.Series({
    "rows where yr != year of dteday": (dates.dt.year - 2011 != train_df["yr"]).sum(),
    "rows where mnth != month of dteday": (dates.dt.month != train_df["mnth"]).sum(),
    "rows where weekday != weekday of dteday (0 = Sunday)":
        ((dates.dt.dayofweek + 1) % 7 != train_df["weekday"]).sum(),
    "rows where workingday != (Mon-Fri) - holiday":
        (mon_to_fri - train_df["holiday"] != train_df["workingday"]).sum(),
    "months that contain more than one season":
        (train_df.groupby("mnth")["season"].nunique() > 1).sum(),
    "rows with weathersit = 4": (train_df["weathersit"] == 4).sum(),
})
print(encoding_facts.to_string())
print(f"correlation of temp and atemp in train: {train_df['temp'].corr(train_df['atemp']):.3f}")
print("day of month in train:", dates.dt.day.min(), "to", dates.dt.day.max(),
      "| in test.csv:", sorted(pd.to_datetime(test_X["dteday"]).dt.day.unique()))

In [ ]:
# Compare the two seasonal representations, with and without the elapsed-time
# trend from dteday. Every design uses the same GD with step 1/largest curvature.
MAX_ITER = 50_000
TOL = 1e-4

design_rows = []
for seasonal in ("season", "mnth"):
    for use_trend in (False, True):
        candidate = Phase1Features(seasonal=seasonal, use_trend=use_trend).fit(train_df)
        X_tr, X_va = candidate.transform(train_df), candidate.transform(val_df)
        eigenvalues = np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr))
        model = LinearRegressionGD(learning_rate=1.0 / eigenvalues[-1],
                                   max_iter=MAX_ITER, tol=TOL)
        result = model.fit(X_tr, y_train)
        val_scores = regression_metrics(y_val, model.predict(X_va))
        design_rows.append({
            "seasonal": seasonal,
            "days_since_start": use_trend,
            "n_features": X_tr.shape[1] - 1,
            "condition_number": eigenvalues[-1] / eigenvalues[0],
            "status": result.status,
            "iterations": result.iterations,
            "train_r2": regression_metrics(y_train, model.predict(X_tr))["r2"],
            "val_r2": val_scores["r2"],
            "val_rmse": val_scores["rmse"],
        })
design_table = pd.DataFrame(design_rows)
design_table

### Learning rate, stopping rule and convergence evidence

**Loss and gradient.** $L(w) = \frac{1}{2n}\lVert Xw - y\rVert^2$, with gradient $\nabla L = \frac{1}{n}X^\top(Xw - y)$, computed explicitly in `LinearRegressionGD.loss_and_gradient`. Initialization: all weights zero. The run is deterministic, so re-running this notebook reproduces our weights exactly.

**Learning rate.** The loss is quadratic, so GD is stable exactly when the learning rate is below $2/\lambda_{max}$, where $\lambda_{max}$ is the largest eigenvalue of $X^\top X/n$ (the largest curvature of the loss). We therefore try multiples of $1/\lambda_{max}$ on both sides of that limit: 0.005, 0.1, 0.5, 1, 1.9 and 2.1. Rule: among the runs that converge, take the one that needs the fewest updates.

**Stopping rule** (the same for every run):
- *converged*: largest |gradient entry| < tol = 1e-4. At the minimum the gradient is exactly zero, so a small gradient is the proof of convergence. A flat loss curve is not: a run that crawls also looks flat.
- *diverged*: the loss becomes non-finite or exceeds 10⁶ × the starting loss.
- *cap*: 50,000 updates.

**Evidence** (outputs below):
- 2.1/λ_max **diverges**: stopped after 84 updates with the loss at 3.7·10¹⁰.
- 0.005/λ_max is **too slow**: it hits the 50,000-update cap with max |gradient| 0.0104, about 100× above tol, although its loss curve already looks flat.
- The four rates in between all **converge** to the same loss (4933.995). 1.9/λ_max (learning rate 0.9551) needs only 267 updates and its loss never increases, so it is our choice.
- tol = 1e-4 is tight enough: GD's validation predictions are within 0.0074 bikes of the exact least-squares minimum. That is an independent check, not used for training.

In [ ]:
# Chosen representation (justified above): month dummies, no elapsed-time trend.
SEASONAL, USE_TREND = "mnth", False
features = Phase1Features(seasonal=SEASONAL, use_trend=USE_TREND).fit(train_df)
X_train, X_val = features.transform(train_df), features.transform(val_df)

lam_max = largest_curvature(X_train)
print(f"{X_train.shape[1] - 1} features + intercept; largest curvature = {lam_max:.4f}; "
      f"GD is stable only for learning rate < 2/curvature = {2 / lam_max:.4f}")

# Learning rates as multiples of 1/curvature, on both sides of the 2/curvature limit.
MULTIPLIERS = [0.005, 0.1, 0.5, 1.0, 1.9, 2.1]
sweep = {}
for multiplier in MULTIPLIERS:
    model = LinearRegressionGD(learning_rate=multiplier / lam_max, max_iter=MAX_ITER, tol=TOL)
    sweep[f"{multiplier:g}/curvature"] = model.fit(X_train, y_train)

sweep_table = pd.DataFrame([{
    "step": label,
    "learning_rate": result.learning_rate,
    "status": result.status,
    "iterations": result.iterations,
    "loss_ever_increased": bool(np.any(np.diff(result.loss_history) > 0)),
    "final_loss": result.loss_history[-1],
    "final_max_abs_gradient": result.grad_history[-1],
} for label, result in sweep.items()])
display(sweep_table)
plot_gd_runs(sweep, TOL, "Phase 1 learning-rate sweep (MSE loss, training portion)")
plt.show()

In [ ]:
# Final Phase 1 model: the converged learning rate that needed the fewest updates.
converged = {label: r for label, r in sweep.items() if r.converged}
chosen_step = min(converged, key=lambda label: converged[label].iterations)
phase1_model = LinearRegressionGD(learning_rate=converged[chosen_step].learning_rate,
                                  max_iter=MAX_ITER, tol=TOL)
phase1_result = phase1_model.fit(X_train, y_train)
assert phase1_result.converged

phase1_metrics = {
    "train": regression_metrics(y_train, phase1_model.predict(X_train)),
    "validation": regression_metrics(y_val, phase1_model.predict(X_val)),
}
print(f"step {chosen_step}: learning rate {phase1_result.learning_rate:.5f}, "
      f"{phase1_result.iterations} updates, final max|gradient| "
      f"{phase1_result.grad_history[-1]:.2e}")
display(pd.DataFrame(phase1_metrics).T)

# Independent check (not used for training): the exact least-squares minimum.
# If GD had stalled, its predictions would differ noticeably from these.
exact_weights = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
gap = np.max(np.abs(X_val @ phase1_result.weights - X_val @ exact_weights))
print(f"largest validation prediction gap between GD and exact minimum: {gap:.4f} bikes")

save_phase1_artifact(
    config.PHASE1_ARTIFACT, features, phase1_result,
    settings={"seed": seed, "max_iter": MAX_ITER, "tol": TOL,
              "step": chosen_step, "largest_curvature": lam_max},
    metrics=phase1_metrics,
)
print("Saved for Phase 2:", config.PHASE1_ARTIFACT.relative_to(config.ROOT))

### Operator check: errors at busy hours

R² treats every hour alike, but the operator's costly failure is an empty station at a busy time. So for every fitted model we also report, on validation:
- errors per decile of demand (decile edges from training demand);
- the **top-decile shortfall**, i.e. the mean bikes missing in hours whose actual demand is above the training 90th percentile;
- the hour × day-type cells where the shortfall concentrates.

Grouping by *actual* demand answers the operator's question directly, but it flatters no model: the top bin also collects hours that were busy partly by chance, so any imperfect model looks under-predicted there. Grouping by *predicted* demand has no such selection effect. Under-prediction that persists there is a genuine bias.

In [ ]:
from src.operator_checks import (
    demand_decile_edges,
    errors_by_demand_decile,
    errors_by_hour_daytype,
    top_decile_shortfall,
    unscaled_coefficients,
)
from src.visualization import plot_shortfall_by_hour

# Operator check: are the busiest hours under-supplied? Decile edges come from
# training demand; error = predicted - actual, shortfall = bikes missing.
decile_edges = demand_decile_edges(y_train)
pred_phase1_val = phase1_model.predict(X_val)
print("training demand decile edges (bikes):", decile_edges[1:-1])
print(f"top-decile shortfall (actual > {decile_edges[-2]:.0f} bikes): "
      f"{top_decile_shortfall(y_val, pred_phase1_val, decile_edges):.1f} bikes per hour")
print("validation hours predicted below zero:", int(np.sum(pred_phase1_val < 0)))

print("\nBy ACTUAL demand decile:")
display(errors_by_demand_decile(y_val, pred_phase1_val, decile_edges).round(2))
print("By PREDICTED demand decile (no selection effect):")
display(errors_by_demand_decile(y_val, pred_phase1_val, decile_edges, bin_on="predicted").round(2))

phase1_hour_errors = errors_by_hour_daytype(val_df, y_val, pred_phase1_val)
print("Hour x day-type cells with the largest shortfall:")
display(phase1_hour_errors.sort_values("mean_shortfall_bikes", ascending=False).head(10).round(1))
plot_shortfall_by_hour({"Phase 1 (MSE)": phase1_hour_errors},
                       "Phase 1: mean bikes short per hour (validation)")
plt.show()

### Outcome

_Fill this from the actual pipeline results._

## Phase 1 bonus — asymmetric cost (optional, +5%)

This section is separate from the chain. It reads the Phase 1 artifact but writes nothing that Phase 2 or the submission file reads (Check 3 below proves the artifact is unchanged). It uses the same split (seed 36954), the same 47 features and fitted scaler, and the same `LinearRegressionGD` loop. Only the loss/gradient hook is replaced (`AsymmetricLinearRegressionGD` in `src/bonus_asymmetric.py`).

**Our reading of "under-predicting costs 3× as much".** We keep the squared error and weight under-predictions by $k = 3$:

$$\ell(r) = \tfrac{1}{2}\,c(r)\,r^2,\qquad r = \hat y - y,\qquad c(r) = \begin{cases} k & r < 0 \text{ (under-prediction)}\\ 1 & r \ge 0 \end{cases}$$

Why this reading:
- The brief asks us to minimize the loss *instead of MSE*, derive *its gradient*, and compare against *our MSE model*. Keeping the quadratic shape changes exactly one thing, the asymmetry, so the shift we measure is caused by the asymmetry alone.
- The loss is differentiable everywhere, so our Phase 1 GD and its gradient-based stopping rule work unchanged.

The other reading, a cost linear in the number of bikes (pinball loss, as in the newsvendor problem, whose optimum is the 75th percentile), is arguably closer to how lost rentals add up. But it has no gradient at $r = 0$, and its slope never shrinks toward zero, so fixed-step GD with our stopping rule could not converge on it. Our assumption is stated openly: a large shortfall is penalized more than in proportion to its size.

**Gradient derivation.**

$$L_k(w) = \frac{1}{2n}\sum_{i=1}^{n} c_i\, r_i^2,\qquad r_i = x_i^\top w - y_i$$

1. *Derivative of one row's loss.* For $r < 0$, $\ell(r) = \tfrac{k}{2}r^2$, so $\ell'(r) = k\,r$. For $r > 0$, $\ell'(r) = r$. At $r = 0$ both one-sided difference quotients $(\ell(h) - \ell(0))/h = \tfrac{1}{2}c\,h$ tend to $0$, so $\ell'(0) = 0$. Hence $\ell'(r) = c(r)\,r$ everywhere, and it is continuous.
2. *Chain rule.* With $\partial r_i/\partial w = x_i$:
$$\nabla L_k(w) = \frac{1}{n}\sum_{i=1}^{n} c_i\, r_i\, x_i = \frac{1}{n}X^\top (c \odot r)$$
3. *Comparison with MSE.* Since $c(r)\,r = r + (k-1)\min(r, 0)$:
$$\nabla L_k = \underbrace{\tfrac{1}{n}X^\top r}_{\text{MSE gradient}} + \tfrac{k-1}{n}X^\top \min(r, 0)$$
This is the MSE gradient plus an extra push that comes only from under-predicted rows. With $k = 1$ it is exactly Phase 1.
4. *Convexity.* $\ell'$ is continuous and non-decreasing (slope $k$ on the left of 0, slope 1 on the right), so $\ell$ is convex. $L_k$ is an average of convex functions of $w$, so any point with zero gradient is the global minimum.
5. *Step size.* $|\ell'(a) - \ell'(b)| \le k\,|a - b|$, so the gradient of $L_k$ changes at most $k$ times as fast as in the MSE case, and GD is guaranteed to converge for learning rates below $2/(k\,\lambda_{max})$. The safe limit is $k$ times smaller than for MSE.
6. *What the minimum means.* The intercept row of $X^\top(c \odot r) = 0$ gives $\sum_{r_i>0} r_i = k\sum_{r_i<0}|r_i|$: on training, total over-prediction equals 3× total under-prediction. For an intercept-only model the minimizer is the 0.75 *expectile* ($\tau = k/(k+1)$), an asymmetric mean, not the 75th percentile.

### Expectation (bonus)

_Write this **before** running the bonus cells._

In [ ]:
from src.bonus_asymmetric import (
    AsymmetricLinearRegressionGD,
    finite_difference_gradient,
    prediction_summary,
    residual_structure,
    shift_by_hour,
)

UNDER_COST = 3.0  # k: under-predicting costs 3x as much as over-predicting

# Read Phase 1's artifact (read-only) and fingerprint it so we can prove later
# that the bonus did not change what Phase 2 consumes.
artifact_hash_before = hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest()
phase1 = load_phase1_artifact(config.PHASE1_ARTIFACT)
bonus_features = Phase1Features.from_dict(phase1["features"])
Xb_train, Xb_val = bonus_features.transform(train_df), bonus_features.transform(val_df)
w_phase1 = phase1["weights"]
assert np.array_equal(Xb_train, X_train), "bonus must use exactly the Phase 1 features"

# Check 1: analytic gradient X^T (c * r) / n against central finite differences.
checker = AsymmetricLinearRegressionGD(under_cost=UNDER_COST)
for name, w in (("at Phase 1 weights", w_phase1), ("at zero weights", np.zeros_like(w_phase1))):
    analytic = checker.loss_and_gradient(Xb_train, y_train, w)[1]
    numeric = finite_difference_gradient(checker, Xb_train, y_train, w)
    error = np.max(np.abs(analytic - numeric)) / np.max(np.abs(analytic))
    print(f"Check 1 {name}: largest relative gradient error = {error:.1e}")

# Check 2: with k = 1 the bonus model is the Phase 1 model, so it must
# reproduce the Phase 1 weights exactly from the same start and settings.
k_one = AsymmetricLinearRegressionGD(
    under_cost=1.0, learning_rate=phase1["learning_rate"],
    max_iter=phase1["settings"]["max_iter"], tol=phase1["settings"]["tol"],
).fit(Xb_train, y_train)
print("Check 2: k = 1 reproduces the Phase 1 weights exactly:",
      np.array_equal(k_one.weights, w_phase1) and k_one.iterations == phase1["iterations"])

**Learning rate, start and stopping rule.**
- *Stopping rule and tol:* identical to Phase 1.
- *Start:* the Phase 1 weights. The feature space is the same, and by convexity (step 4) the start does not change the answer.
- *Learning rate:* step 5 says the safe limit shrinks by $k$. We run the Phase 1 learning rate unchanged, and the Phase 1 learning rate divided by $k$. The second is 0.955/3 = 0.318, below the guaranteed limit $2/(3\lambda_{max}) = 0.335$.

In [ ]:
# Learning rate: the Phase 1 step, and the Phase 1 step divided by k (the
# gradient can change up to k times faster, so the safe step shrinks by k).
# Both start from the Phase 1 weights.
phase1_step = phase1["learning_rate"]
bonus_runs = {}
for label, step in (("Phase 1 step", phase1_step), ("Phase 1 step / k", phase1_step / UNDER_COST)):
    model = AsymmetricLinearRegressionGD(under_cost=UNDER_COST, learning_rate=step,
                                         max_iter=MAX_ITER, tol=TOL)
    bonus_runs[label] = model.fit(Xb_train, y_train, init_weights=w_phase1)

display(pd.DataFrame([{
    "step": label,
    "learning_rate": result.learning_rate,
    "status": result.status,
    "iterations": result.iterations,
    "loss_ever_increased": bool(np.any(np.diff(result.loss_history) > 0)),
    "final_loss": result.loss_history[-1],
    "final_max_abs_gradient": result.grad_history[-1],
} for label, result in bonus_runs.items()]))
plot_gd_runs(bonus_runs, TOL, "Bonus: asymmetric loss (k = 3), warm start from Phase 1 weights")
plt.show()

asym_result = bonus_runs["Phase 1 step / k"]
assert asym_result.converged

In [ ]:
# How the fitted predictions shift (validation set, same rows for both models).
pred_mse = Xb_val @ w_phase1
pred_asym = Xb_val @ asym_result.weights

comparison = pd.DataFrame({
    "MSE model (Phase 1)": prediction_summary(y_val, pred_mse, UNDER_COST),
    "Asymmetric model (k=3)": prediction_summary(y_val, pred_asym, UNDER_COST),
})
comparison["change"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
display(comparison)

shift = pred_asym - pred_mse
print(f"shift per hour (bikes): mean {shift.mean():.1f}, median {np.median(shift):.1f}, "
      f"min {shift.min():.1f}, max {shift.max():.1f}; "
      f"share of hours shifted down: {np.mean(shift < 0):.1%}")

hour_table = shift_by_hour(val_df, y_val, pred_mse, pred_asym)
plot_shift_by_hour(hour_table, "Mean predictions by hour: MSE vs asymmetric (validation)")
plt.show()

config.BONUS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(config.BONUS_DIR / "validation_comparison.csv")
hour_table.to_csv(config.BONUS_DIR / "shift_by_hour.csv", index=False)

In [ ]:
# Underfitting vs demand variability. Phase 1 has no hour x day-type terms, so
# its residuals in each hour x day-type cell need not average to zero.
train_residual = Xb_train @ w_phase1 - y_train
cell_table, shares = residual_structure(train_df, train_residual)
print(f"Phase 1 training residual variance: {shares['between_cell_share']:.1%} between "
      f"hour x day-type cells (systematic misfit), {shares['within_cell_share']:.1%} within cells")

# Where does the shift land? Cells with a positive mean residual are ones the
# MSE model already over-predicts; raising them further only adds cost.
val_cells = (val_df[["workingday", "hr"]].assign(shift=shift)
             .merge(cell_table, on=["workingday", "hr"], how="left"))
already_over = val_cells["mean_residual"] > 0
print(f"validation hours in already over-predicted cells: {already_over.mean():.1%}, "
      f"receiving {val_cells['shift'][already_over].sum() / val_cells['shift'].sum():.1%} "
      f"of the total upward shift")
print(f"mean shift there: {val_cells['shift'][already_over].mean():.1f} bikes, "
      f"elsewhere: {val_cells['shift'][~already_over].mean():.1f} bikes")

cells = hour_table.merge(cell_table, on=["workingday", "hr"])
print(f"correlation across the 48 cells: shift vs within-cell residual SD = "
      f"{cells['shift'].corr(cells['within_sd']):.2f}, "
      f"shift vs cell mean residual = {cells['shift'].corr(cells['mean_residual']):.2f}")
cells.sort_values("shift", ascending=False).head(8).round(1)

In [ ]:
# Operator check for the bonus: does k = 3 fix the busiest hours, or lift everything?
for label, pred in (("MSE model", pred_mse), ("Asymmetric model", pred_asym)):
    print(f"{label}: top-decile shortfall "
          f"{top_decile_shortfall(y_val, pred, decile_edges):.1f} bikes per hour")

by_decile_mse = errors_by_demand_decile(y_val, pred_mse, decile_edges)
by_decile_asym = errors_by_demand_decile(y_val, pred_asym, decile_edges)
display(pd.DataFrame({
    "decile": by_decile_mse["decile"],
    "mean_actual": by_decile_mse["mean_actual"],
    "mean_shift": by_decile_asym["mean_predicted"] - by_decile_mse["mean_predicted"],
    "under_rate_mse": by_decile_mse["under_prediction_rate"],
    "under_rate_asym": by_decile_asym["under_prediction_rate"],
    "shortfall_mse": by_decile_mse["mean_shortfall_bikes"],
    "shortfall_asym": by_decile_asym["mean_shortfall_bikes"],
}).round(2))

# Which coefficients moved? In original units: a dummy's coefficient is the
# bikes it adds relative to its reference level (hr=0, Sunday, January, clear).
names, mean, std = bonus_features.feature_names, bonus_features.mean, bonus_features.std
coefficients = pd.DataFrame({
    "mse": unscaled_coefficients(w_phase1, names, mean, std),
    "asym": unscaled_coefficients(asym_result.weights, names, mean, std),
})
coefficients["change"] = coefficients["asym"] - coefficients["mse"]
largest_moves = coefficients["change"].abs().sort_values(ascending=False).index[:12]
display(coefficients.loc[largest_moves].round(1))

plot_shortfall_by_hour({
    "MSE model": errors_by_hour_daytype(val_df, y_val, pred_mse),
    "Asymmetric model (k=3)": errors_by_hour_daytype(val_df, y_val, pred_asym),
}, "Mean bikes short per hour: MSE vs asymmetric (validation)")
plt.show()

In [ ]:
# Check 3: the bonus left Phase 1's artifact (what Phase 2 consumes) untouched.
assert hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest() == artifact_hash_before
assert np.array_equal(load_phase1_artifact(config.PHASE1_ARTIFACT)["weights"],
                      phase1_result.weights)
print("Check 3: Phase 1 artifact and weights unchanged by the bonus")

### Outcome (bonus)

_Draft written from the run above. Rewrite it in your own words before submission._

**Optimization.**
- The Phase 1 learning rate (0.955) **diverged** after 12 updates under $k = 3$. The Phase 1 rate divided by $k$ (0.318) converged in 388 updates, and its loss never increased.
- All three code checks pass:
  - the analytic gradient matches finite differences (relative error ≤ 2.7·10⁻¹⁰);
  - $k = 1$ reproduces the Phase 1 weights exactly;
  - the Phase 1 artifact is unchanged.

**How the predictions shift** (validation, 2,178 hours):

| | MSE model (Phase 1) | Asymmetric model ($k=3$) |
|---|---:|---:|
| R² | 0.6823 | 0.6059 |
| RMSE | 103.56 | 115.35 |
| asymmetric cost ($k=3$) | 11,145 | 9,343 (−16.2%) |
| mean prediction | 196.4 | 239.0 |
| hours under-predicted | 45.8% | 32.2% |
| mean shortfall (bikes/hour) | 37.2 | 21.2 |
| mean surplus (bikes/hour) | 39.6 | 66.1 |

- Predictions rise by 42.6 bikes/hour on average (median 39.2, range −2.7 to +108.7). Only 0.2% of hours go down.
- For the operator: 16.0 fewer bikes short per hour, at the price of 26.6 more idle bikes per hour. When a missing bike costs 3× an idle one this is a good trade, and the 16% lower asymmetric cost measures it.
- R² and RMSE get worse, as expected, because they score the MSE objective.
- 32.2% of hours are still under-predicted, not 25%. That is consistent with step 6: the loss targets an asymmetric mean, not the 75th percentile.

**How much of the shift is Phase 1 underfitting, and how much is genuine demand variability?**
- **Size of the underfitting.** Phase 1 has only main effects. 44.4% of its training residual variance lies *between* hour × day-type cell means. That is systematic error a model with hour × working-day terms would remove. The remaining 55.6% (*within* cells) is the most that can be genuine variability, and it still contains other missing effects (e.g. hour × temperature). So underfitting accounts for at least 44% of the residual spread the loss reacts to.
- **Underfitting misplaces the shift.** The model cannot raise weekday 08:00 without raising weekend 08:00 too.
  - 50.5% of validation hours fall in cells the MSE model already over-predicts on average. They still receive 48.6% of the total upward shift (+41.0 bikes/hour, vs +44.2 elsewhere).
  - Worst case: weekend 08:00, already over-predicted (383 vs an actual mean of 106), is raised by another 98 to 482.
- **Some of the gain is correcting underfitting, not hedging uncertainty.** MSE under-predicts the working-day peaks (08:00: 374 vs 480; 17:00: 465 vs 533). The asymmetric model lands close to the actual means (456 and 536), mostly because it shifts everything up, not because it learned the peaks.
- **The variability-driven part.** Across the 48 cells, the shift correlates 0.67 with the within-cell residual spread and only 0.14 with the cell's mean residual. The loss responds to spread, not to where the bias is: variable commute hours get the largest buffers (+70 to +98 bikes), working-day late-evening and night hours (21:00–05:00) the smallest (+17 to +29).

**Busiest hours (operator check).**
- The top-decile shortfall (hours above 452 actual bikes) falls from 176.6 to 112.1 bikes per hour (−37%). But 88% of those hours are still under-predicted, versus 96% before.
- The lift grows with demand, but only mildly: +29.5 bikes in the quietest decile, +69.5 in the busiest. Much of it is a broad lift. The raw intercept rises by 36.9 bikes, which applies to every hour, so even 03:00 gets more bikes.
- The targeted part sits in the hour dummies: 08:00 +57.2, 18:00 +44.2, 17:00 +43.5, and weekend-midday hours (12:00–15:00) +18 to +25. The Mon and Fri dummies fall by 17 and 22 bikes relative to Sunday, so the model leans toward weekends. This is the loss chasing Phase 1's largest misses, at weekend 11:00–15:00 (120–154 bikes short) and working-day 08:00/17:00 (107–118 short). With main effects only, it can reach them only by moving whole hours and whole days.

**Conclusion.** With Phase 1's main-effects features, about half of the extra bikes go to hours that were already over-stocked, and that part is an artifact of underfitting. A clean estimate of the buffer that genuine demand variability justifies needs a model without this underfitting, i.e. the same bonus run on Phase 2's features. The asymmetric loss shifts errors toward surplus, but it cannot create structure the features lack.

## Phase 2 — Polynomial Regression

### Which interactions? Residual diagnosis of Phase 1 (training rows only)

Phase 1 has main effects only, so each training residual $e = y - \hat y$ contains whatever the main effects cannot express. For example, if the 08:00 level depends on whether it is a working day, Phase 1's 08:00 residuals will be positive on working days and negative on the others.

For each candidate combination we group the 8,708 training residuals into cells and measure:
- **between share**: the share of Phase 1's residual variance that the cell means explain, i.e. how much of what Phase 1 misses this combination could recover. Overlapping combinations (e.g. month and temperature) are not additive.
- **adjusted share**: the between share corrected for the number of cells, like adjusted R², so a combination with many cells cannot win by size alone.
- **cell support**: rows per cell, and how many cells have fewer than 30 rows (a 30-row mean is still uncertain by about ±18 bikes).

Continuous columns are cut into equal-count bins on training rows: 10 bins for `temp` alone, 5 bins inside products. Validation rows are not used. Phase 1's residuals average to zero within every hour (checked below), so an hour × X cell measures only the interaction.

In [ ]:
from src.phase2_polynomial import residual_cell_means, residual_diagnosis
from src.visualization import plot_residual_profiles

# Phase 2 starts from Phase 1's artifact: its features, scaler and weights.
phase1_artifact = load_phase1_artifact(config.PHASE1_ARTIFACT)
phase1_features = Phase1Features.from_dict(phase1_artifact["features"])
phase1_residual = y_train - phase1_features.transform(train_df) @ phase1_artifact["weights"]
hour_means = pd.Series(phase1_residual).groupby(train_df["hr"].to_numpy()).mean()
print(f"mean training residual {phase1_residual.mean():.4f}; "
      f"largest |mean residual| of any single hour {hour_means.abs().max():.4f}")
print("holiday dates in the training portion:",
      train_df.loc[train_df["holiday"] == 1, "dteday"].nunique())

diagnosis = residual_diagnosis(train_df, phase1_residual)
config.PHASE2_DIR.mkdir(parents=True, exist_ok=True)
diagnosis.to_csv(config.PHASE2_DIR / "residual_diagnosis.csv", index=False)
display(diagnosis.round(4))

panels = [
    ("hour x workingday", residual_cell_means(train_df, phase1_residual, ["hr", "workingday"]),
     "workingday", False),
    ("hour x yr", residual_cell_means(train_df, phase1_residual, ["hr", "yr"]), "yr", False),
    ("temperature quintile x hour",
     residual_cell_means(train_df, phase1_residual, ["temp_bin5", "hr"]), "temp_bin5", True),
    ("hour x season", residual_cell_means(train_df, phase1_residual, ["hr", "season"]),
     "season", False),
]
plot_residual_profiles(panels, "Phase 1 training residuals by hour (positive = under-predicted)")
plt.show()

### Re-ranking after the confirmed groups

The Step A shares overlap (month, season and temperature all follow the yearly cycle), so we re-rank the remaining candidates on what is left *after* the confirmed groups:
1. remove the hour × workingday group means from Phase 1's training residuals;
2. remove hour × workingday and hour × yr together (least squares on both groups' indicators; for one group this is just its group means).

For a continuous candidate (hour × temp, temp × hum, temp²) the share is the R² of the remaining residuals regressed on its columns; for a group candidate this R² is exactly the between-group share of Step A. Training rows only.

In [ ]:
from src.phase2_polynomial import (
    RERANK_CANDIDATES,
    diagnosis_frame,
    project_out,
    remove_group_means,
    rerank,
)

frame = diagnosis_frame(train_df)
phase1_total = np.sum(phase1_residual ** 2)
stages = {
    "after hour x workingday":
        remove_group_means(phase1_residual, frame, [["hr", "workingday"]]),
    "after hour x workingday + hour x yr":
        remove_group_means(phase1_residual, frame, [["hr", "workingday"], ["hr", "yr"]]),
}
for stage, residual in stages.items():
    remaining = np.sum((residual - residual.mean()) ** 2) / phase1_total
    print(f"{stage}: {remaining:.1%} of Phase 1's residual variance remains")
    table = rerank(residual, frame)
    table.insert(3, "share_of_phase1_variance", table["share"] * remaining)
    display(table.round(4))

# Season and temperature both follow the yearly cycle: what does each add after the other?
after_confirmed = stages["after hour x workingday + hour x yr"]
overlap_rows = []
for first, second in [("hour x season", "hour x temp"),
                      ("hour x temp", "hour x season"),
                      ("hour x season", "hour x workingday x temp"),
                      ("hour x season", "hour x workingday x season")]:
    residual = project_out(after_confirmed, frame, RERANK_CANDIDATES[first])
    remaining = np.sum((residual - residual.mean()) ** 2) / phase1_total
    added = rerank(residual, frame, {second: RERANK_CANDIDATES[second]}).iloc[0]
    overlap_rows.append({
        "already in": first,
        "candidate": second,
        "adds_share_of_phase1_variance": added["share"] * remaining,
        "adjusted_share_of_remaining": added["adjusted_share"],
    })
display(pd.DataFrame(overlap_rows).round(4))

### Phase 2 design: which terms, how they are formed, how they are fitted

**Groups, in the order they are added** (evidence: the two tables above).

| Degree | Group | Columns | Why |
|---|---|---:|---|
| 2 | hour × workingday | 23 | Largest Step A share (0.444); the weekday/weekend daily shape. |
| 2 | hour × yr | 23 | 0.107 in Step A; 2012 peaks are higher, not just shifted. |
| 2 | hour × season | 69 | Best yearly-cycle candidate after the confirmed groups (adjusted 0.2055 vs hour × temp 0.1822). After it, hour × temp adds only 0.0061 of Phase 1's variance, while season still adds 0.0283 after temp. No sparse cells (min 77 rows). Month was rejected in Step A (105 sparse cells). |
| 2 | hour-block × weathersit | 8 | Day parts: night 0–5, morning 6–9, midday 10–15, evening 16–19, late 20–23. Adjusted 0.0373 with 15 cells and none sparse, vs full hour × weathersit 0.0420 with 72 cells, 13 of them sparse. |
| 2 | temp × hum | 1 | Adjusted 0.0323 for a single extra column (temp and hum are already in Phase 1). |
| 3 | hour × workingday × yr | 24 | Still 0.0525 adjusted after both confirmed groups: the weekday peaks grow more from 2011 to 2012 than the rest of the day. Includes workingday × yr. Min 117 rows per term. |
| 3 | hour × workingday × temp | 47 | Adds 0.0246 after hour × season with no sparse cells, more than hour × workingday × season (0.0229, 59 sparse cells). Includes hour × temp and workingday × temp, so the model stays hierarchical. |

Rejected after re-ranking: hour × temp as a separate degree-2 group (overlaps season; it enters only inside the degree-3 group), full hour × weathersit (sparse), temp² (adjusted 0.0029).

**How terms are formed.**
- Explicit term lists only (`group_terms` in `src/phase2_polynomial.py`). No product of two levels of one column (always 0), no square of a 0/1 column, and reference levels are skipped, so no term duplicates a Phase 1 column.
- `workingday` re-enters only through its interactions; its main effect is already carried by `weekday` + `holiday`.
- A product is built from raw values. The hour (or hour-block) indicator is used as 0/1; every other factor is first centered at its training mean (e.g. `hr=8 * (workingday − its training mean)`). Centering does not change which models the columns can express, but it makes the products much less correlated with the hour indicators: the full design's condition number with and without centering is shown below.
- Phase 1's 47 columns keep Phase 1's fitted scaler unchanged. Each new column is standardized with its own training mean and standard deviation.

**How it is fitted.**
- Degree = interaction order. Degree 1 is Phase 1 exactly; degree 2 adds the five pair groups; degree 3 adds the two triple groups. Groups are added one at a time and every step is scored, so each group's contribution is a number.
- Every step starts from Phase 1's weights in their own columns and 0 for every new column (the brief's warm start). Before training we assert that this starting model reproduces Phase 1's predictions.
- Same GD code, loss, stopping rule (max |gradient| < 1e-4, cap 50,000) and learning-rate rule as Phase 1: the same multiple of 1/λ_max, with λ_max recomputed for each expanded matrix.
- Degree is chosen on validation R²; the top-decile shortfall (by actual and by predicted demand) is reported next to it for the operator.

In [ ]:
from src.phase2_polynomial import PHASE2_GROUPS, Phase2Features

# Design checks on training rows only (nothing is fitted here).
phase1_train_pred = phase1_features.transform(train_df) @ phase1_artifact["weights"]
design_rows, groups_so_far = [], []
for group in [None, *PHASE2_GROUPS]:
    if group:
        groups_so_far.append(group)
    candidate = Phase2Features(phase1_features, groups_so_far).fit(train_df)
    X_tr = candidate.transform(train_df)
    eigenvalues = np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr))
    start = X_tr @ candidate.warm_start(phase1_artifact["weights"])
    design_rows.append({
        "added group": group or "(Phase 1)",
        "degree": PHASE2_GROUPS.get(group, 1),
        "columns": X_tr.shape[1],
        "rank": np.linalg.matrix_rank(X_tr),
        "condition_number": eigenvalues[-1] / eigenvalues[0],
        "warm_start_max_prediction_change": np.max(np.abs(start - phase1_train_pred)),
    })
display(pd.DataFrame(design_rows))

uncentered = Phase2Features(phase1_features, list(PHASE2_GROUPS), center_factors=False).fit(train_df)
X_unc = uncentered.transform(train_df)
eig_unc = np.linalg.eigvalsh(X_unc.T @ X_unc / len(X_unc))
print(f"full design condition number: {design_rows[-1]['condition_number']:.0f} with centering, "
      f"{eig_unc[-1] / eig_unc[0]:.0f} without")

print("Training rows supporting each new term (non-zero before centering):")
display(candidate.term_support(train_df).groupby("group", sort=False)["support_rows"]
        .agg(terms="count", min_rows="min", median_rows="median"))

### Expectation

_Write this **before** running the phase._

In [ ]:
from src.phase2_polynomial import (
    PHASE2_GROUPS,
    Phase2Features,
    fit_and_score,
    save_phase2_artifact,
)

# Same rule as Phase 1: the same multiple of 1/largest curvature, same stopping rule.
settings1 = phase1_artifact["settings"]
STEP_MULTIPLIER = phase1_artifact["learning_rate"] * settings1["largest_curvature"]
print(f"learning rate = {STEP_MULTIPLIER:.2f} / largest curvature; "
      f"tol {settings1['tol']}, cap {settings1['max_iter']} updates")

# Add the groups one at a time; every step starts from the Phase 1 weights.
sequence_rows, fitted_steps, groups_so_far = [], {}, []
for group in [None, *PHASE2_GROUPS]:
    if group:
        groups_so_far.append(group)
    step_features = Phase2Features(phase1_features, groups_so_far).fit(train_df)
    result, row = fit_and_score(step_features, train_df, y_train, val_df, y_val,
                                phase1_artifact["weights"], STEP_MULTIPLIER,
                                settings1["max_iter"], settings1["tol"], decile_edges)
    label = group or "(Phase 1)"
    fitted_steps[label] = (step_features, result)
    sequence_rows.append({"added group": label, "degree": PHASE2_GROUPS.get(group, 1), **row})
sequence = pd.DataFrame(sequence_rows)
config.PHASE2_DIR.mkdir(parents=True, exist_ok=True)
sequence.to_csv(config.PHASE2_DIR / "phase2_sequence.csv", index=False)

print("Scores after each added group (validation unless marked train):")
display(sequence[["added group", "degree", "n_features", "train_r2", "val_r2",
                  "train_val_r2_gap", "val_rmse", "top_decile_shortfall",
                  "top_predicted_decile_shortfall", "top_predicted_decile_mean_error",
                  "negative_predictions"]].round(4))
print("Convergence of each fit:")
display(sequence[["added group", "rank_ok", "condition_number", "learning_rate", "status",
                  "iterations", "final_max_abs_gradient", "gap_to_exact_bikes"]])

In [ ]:
# A degree's model is the last step of that degree (all of its groups added).
degree_rows = {degree: sequence[sequence["degree"] == degree].iloc[-1] for degree in (1, 2, 3)}
chosen_degree = max(degree_rows, key=lambda degree: degree_rows[degree]["val_r2"])
chosen_row = degree_rows[chosen_degree]
phase2_features, phase2_result = fitted_steps[chosen_row["added group"]]
display(pd.DataFrame(degree_rows).T[["added group", "n_features", "train_r2", "val_r2",
                                     "train_val_r2_gap", "val_rmse", "top_decile_shortfall",
                                     "top_predicted_decile_shortfall", "status", "iterations"]])
print(f"Chosen degree (highest validation R2): {chosen_degree}")

# Operator checks for the three degree models.
degree_predictions = {}
for degree, row in degree_rows.items():
    step_features, step_result = fitted_steps[row["added group"]]
    degree_predictions[f"degree {degree}"] = step_features.transform(val_df) @ step_result.weights

def by_decile(bin_on, column):
    """One column per degree model; a decile no hour falls into stays empty."""
    columns = {}
    for label, pred in degree_predictions.items():
        table = errors_by_demand_decile(y_val, pred, decile_edges, bin_on=bin_on)
        columns[label] = pd.Series(table[column].to_numpy(), index=table["decile"].astype(int))
    return pd.DataFrame(columns).rename_axis("decile")

print("Mean shortfall (bikes) by ACTUAL demand decile:")
display(by_decile("actual", "mean_shortfall_bikes").round(1))
print("Under-prediction rate by ACTUAL demand decile:")
display(by_decile("actual", "under_prediction_rate").round(2))
print("Mean error (predicted - actual) by PREDICTED demand decile (no selection effect):")
display(by_decile("predicted", "mean_error").round(1))
print("Mean shortfall (bikes) by PREDICTED demand decile:")
display(by_decile("predicted", "mean_shortfall_bikes").round(1))

hour_errors = {label: errors_by_hour_daytype(val_df, y_val, pred)
               for label, pred in degree_predictions.items()}
print(f"Hour x day-type cells with the largest shortfall, degree {chosen_degree}:")
display(hour_errors[f"degree {chosen_degree}"]
        .sort_values("mean_shortfall_bikes", ascending=False).head(10).round(1))
plot_shortfall_by_hour(hour_errors, "Mean bikes short per hour by degree (validation)")
plt.show()

save_phase2_artifact(
    config.PHASE2_ARTIFACT, phase2_features, phase2_result, chosen_degree,
    chosen_row.to_dict(), sequence,
    settings={"seed": seed, "step_multiplier": STEP_MULTIPLIER,
              "max_iter": settings1["max_iter"], "tol": settings1["tol"]},
)
print("Saved for Phase 3:", config.PHASE2_ARTIFACT.relative_to(config.ROOT))

### Outcome

_Fill this from the actual pipeline results._

## Phase 3 — Bias–Variance Tradeoff

### Expectation

_Write this **before** running the phase._

In [ ]:
# Phase 3 experiment code goes here

### Outcome

_Fill this from the actual pipeline results._

## Phase 4 — Regularization

### Expectation

_Write this **before** running the phase._

In [ ]:
# Phase 4 experiment code goes here

### Outcome

_Fill this from the actual pipeline results._

## Phase 5 — Logistic Regression

### Expectation

_Write this **before** running the phase._

In [ ]:
# Phase 5 experiment code goes here

### Outcome

_Fill this from the actual pipeline results._